# 7 · End-to-End Pipeline + Observability

Wire the pieces into one `process_ticket()` function and trace a full run with **MLflow**.

Pipeline: **intake chain → routing chain → supervisor (specialists) → draft chain → summary chain**,
with the action agent + HITL for refunds. This notebook is the capstone integration point.

In [1]:
%run aurora_common.py

C:\Users\akhawaja\git\cs4603\wk4_capstone\aurora_common.py:42: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Turn on MLflow tracing
Open the MLflow UI (`mlflow ui --port 5000`) to see one trace per ticket.

In [2]:
mlflow = enable_mlflow_tracing("aurora-capstone")
print("MLflow experiment set: aurora-capstone")

2026/07/01 16:32:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/01 16:32:04 INFO mlflow.store.db.utils: Updating database tables


2026/07/01 16:32:06 INFO mlflow.tracking.fluent: Experiment with name 'aurora-capstone' does not exist. Creating a new experiment.


MLflow experiment set: aurora-capstone


## Assemble a minimal pipeline
> **TODO (student):** import/reuse your chains and agents from notebooks 1–5 instead of the compact
> inline versions below. Add guardrails/PII and the refund HITL path.

In [3]:
from pydantic import BaseModel, Field
from typing import Optional, Literal

retriever = get_knowledge_retriever(k=3)
orders_db = get_orders_sqldatabase()


class Routing(BaseModel):
    route_to: Literal["knowledge", "order"] = Field(description="Which specialist handles this")


routing_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Route the ticket: policy questions -> knowledge, order/customer data -> order."),
        ("human", "{ticket}"),
    ])
    | llm_noreason.with_structured_output(Routing)
)


@tool
def search_help_center(query: str) -> str:
    """Search store help-center policies."""
    return "\n\n".join(d.page_content for d in retriever.invoke(query))


@tool
def run_sql(query: str) -> str:
    """Run a read-only SQL query against the orders database."""
    try:
        return orders_db.run(query)
    except Exception as e:
        return f"Error: {e}"


knowledge_agent = create_agent(model=llm_noreason, tools=[search_help_center],
                               system_prompt="Answer policy questions using the help-center tool.")
order_agent = create_agent(model=llm_noreason, tools=[run_sql],
                           system_prompt="Answer order questions with SELECT queries. Tables: customers, products, orders, order_items.")

draft_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are Aurora, a friendly support agent. Write a concise reply from the research."),
        ("human", "Customer: {ticket}\n\nResearch: {research}\n\nReply:"),
    ])
    | llm_noreason | StrOutputParser()
)

In [4]:
@mlflow.trace(name="process_ticket")
def process_ticket(ticket: str) -> str:
    routing = routing_chain.invoke({"ticket": ticket})
    specialist = knowledge_agent if routing.route_to == "knowledge" else order_agent
    research = specialist.invoke({"messages": [HumanMessage(content=ticket)]})["messages"][-1].content
    return draft_chain.invoke({"ticket": ticket, "research": research})


for t in [
    "What is the status of order 1001?",
    "How many days do I have to return opened electronics?",
]:
    print("TICKET:", t)
    print("REPLY :", process_ticket(t))
    print()

TICKET: What is the status of order 1001?


REPLY : Hello! Order 1001 has been shipped and is on its way to you. You should receive a tracking update shortly. Let me know if you need anything else!

TICKET: How many days do I have to return opened electronics?


REPLY : Hello! You have **14 days** to return opened electronics.

Please note that while standard returns for unused items in original packaging are accepted within 30 days, Premium members enjoy an extended 60-day return window. Let me know if you need any further assistance!



### Definition of done
- `process_ticket()` runs intake→route→specialist→draft end-to-end.
- Each run appears as an MLflow trace with nested spans.
- Capstone: extend with the refund HITL path, PII redaction, retries, and the QA/summary chain, and
  demo three tickets (order status, policy, refund > $100 requiring approval).